<a href="https://colab.research.google.com/github/sampada-11/Data_Science_Lab/blob/main/exp_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load the dataset
df = pd.read_csv('/content/HousingData.csv')

print("First 5 rows of the dataset:")
print(df.head())
print("\nColumns in the dataset:")
print(df.columns)

# --- Feature and Target Variable Selection ---
# Adjusted based on the actual columns in HousingData.csv

# Numerical features representing 'size', 'number of rooms', and some 'location' aspects
numerical_features = ['RM', 'LSTAT', 'CRIM', 'DIS', 'TAX', 'PTRATIO', 'NOX'] # RM (rooms), LSTAT (lower status pop.), CRIM (crime rate), DIS (employment distance), TAX (property tax), PTRATIO (pupil-teacher ratio), NOX (nitric oxides concentration)

# Categorical features representing 'location' (CHAS is a binary indicator for Charles River)
categorical_features = ['CHAS']

# Target variable (house prices)
target_variable = 'MEDV' # Median value of owner-occupied homes

# Combine all features
all_features = numerical_features + categorical_features

# Check if selected columns exist in the DataFrame
missing_cols = [col for col in all_features + [target_variable] if col not in df.columns]
if missing_cols:
    print(f"\nError: The following selected columns are not found in your dataset: {missing_cols}")
    print("Please update the 'numerical_features', 'categorical_features', and 'target_variable' to match your dataset's actual column names.")
else:
    # Separate features (X) and target (y)
    X = df[all_features]
    y = df[target_variable]

    # --- Preprocessing Pipeline ---
    # Impute missing numerical values with the mean, then scale
    numerical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])

    # Impute missing categorical values with the most frequent, then one-hot encode
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    # Create a preprocessor to apply different transformations to different columns
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_features),
            ('cat', categorical_transformer, categorical_features)
        ])

    # --- Build the Linear Regression model pipeline ---
    model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                   ('regressor', LinearRegression())])

    # --- Split the data into training and testing sets ---
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # --- Train the model ---
    print("\nTraining the Linear Regression model...")
    model_pipeline.fit(X_train, y_train)
    print("Model training complete.")

    # --- Make predictions on the test set ---
    y_pred = model_pipeline.predict(X_test)

    # --- Evaluate the model ---
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"\nModel Evaluation:")
    print(f"Mean Absolute Error (MAE): {mae:.2f}")
    print(f"R-squared (R2): {r2:.2f}")


First 5 rows of the dataset:
      CRIM    ZN  INDUS  CHAS    NOX     RM   AGE     DIS  RAD  TAX  PTRATIO  \
0  0.00632  18.0   2.31   0.0  0.538  6.575  65.2  4.0900    1  296     15.3   
1  0.02731   0.0   7.07   0.0  0.469  6.421  78.9  4.9671    2  242     17.8   
2  0.02729   0.0   7.07   0.0  0.469  7.185  61.1  4.9671    2  242     17.8   
3  0.03237   0.0   2.18   0.0  0.458  6.998  45.8  6.0622    3  222     18.7   
4  0.06905   0.0   2.18   0.0  0.458  7.147  54.2  6.0622    3  222     18.7   

        B  LSTAT  MEDV  
0  396.90   4.98  24.0  
1  396.90   9.14  21.6  
2  392.83   4.03  34.7  
3  394.63   2.94  33.4  
4  396.90    NaN  36.2  

Columns in the dataset:
Index(['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX',
       'PTRATIO', 'B', 'LSTAT', 'MEDV'],
      dtype='object')

Training the Linear Regression model...
Model training complete.

Model Evaluation:
Mean Absolute Error (MAE): 3.32
R-squared (R2): 0.64
